# Notebook 08: Final Pipeline & Model Comparison
Complete model training, evaluation, comparison, and interpretability analysis

## Cell 1 — Imports

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))
from src.data.data_loader import (
    load_config,
    load_dataset,
    sample_dataset
)
from src.data.preprocessing import (
    basic_preprocessing_pipeline
)
from src.features.feature_engineering import (
    feature_engineering_pipeline
)
from src.features.encoder import (
    encode_categorical_columns
)
from src.models.train_test_split import (
    temporal_train_test_split,
    split_features_target
)
from src.models.smote_pipeline import (
    apply_smote
)
from src.models.boosting_models import (
    train_xgboost,
    train_lightgbm,
    train_catboost
)
from src.models.baseline_models import (
    train_logistic_regression,
    train_random_forest,
    generate_predictions
)
from src.evaluation.model_comparison import (
    ModelComparison
)
from src.visualization.shap_visualizer import (
    create_tree_explainer,
    compute_shap_values,
    plot_shap_summary
)
from src.visualization.temporal_shap import (
    compute_yearly_shap_importance,
    plot_temporal_feature_drift
)
from src.visualization.save_figures import (
    save_current_figure
)

## Cell 2 — Load and Prepare Dataset

In [2]:
config = load_config()
df = load_dataset(config)
df = sample_dataset(df, config)
df = basic_preprocessing_pipeline(df)
df = feature_engineering_pipeline(df)
train_df, test_df = temporal_train_test_split(df)
X_train, X_test, y_train, y_test = split_features_target(
    train_df,
    test_df
)
X_train, X_test, encoders = encode_categorical_columns(
    X_train,
    X_test
)
X_train_smote, y_train_smote = apply_smote(
    X_train,
    y_train
)

print(f"✓ Data prepared successfully!")
print(f"  Training set (SMOTE): {X_train_smote.shape}")
print(f"  Test set: {X_test.shape}")
print(f"  Features: {X_train.shape[1]}")


Loading dataset from:
/Users/nurnafisfuad/Desktop/AccidentXAI/data/raw/US_Accidents_March23.csv


Dataset loaded successfully.


Sampling 100000 rows...


Starting preprocessing pipeline...

Converting Start_Time to datetime format...
Removed 85 duplicate rows.

Selected relevant columns.

Dropping high-missing columns:

[]

Missing values handled successfully.

Preprocessing completed successfully.


Starting feature engineering pipeline...

Creating temporal features...

Creating rush-hour feature...

Creating night-driving feature...

Simplifying target variable...


Feature engineering completed.


Performing temporal split...

Train Shape: (71919, 30)
Test Shape: (16318, 30)

Separating features and target...

X_train shape: (71919, 29)
X_test shape: (16318, 29)

Encoding categorical columns...

Encoding completed.


Applying SMOTE...

Before SMOTE:

Counter({0: 54138, 1: 17781})

After SMOTE:

Counter({0: 54138, 1: 54138})
✓ Data prepared successfully!
  Training set (SMOTE): (1

## Cell 3 — Initialize Comparison Engine

In [3]:
comparison_engine = ModelComparison()
print("✓ Model comparison engine initialized")

✓ Model comparison engine initialized


## Cell 4 — Logistic Regression

In [4]:
log_model = train_logistic_regression(
    X_train_smote,
    y_train_smote
)
log_preds, log_probs = generate_predictions(
    log_model,
    X_test
)
comparison_engine.add_model_result(
    "Logistic Regression",
    y_test,
    log_preds,
    log_probs
)
print("✓ Logistic Regression completed")


Training Logistic Regression...

Logistic Regression training completed.

✓ Logistic Regression completed


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Cell 5 — Random Forest

In [5]:
rf_model = train_random_forest(
    X_train_smote,
    y_train_smote
)
rf_preds, rf_probs = generate_predictions(
    rf_model,
    X_test
)
comparison_engine.add_model_result(
    "Random Forest",
    y_test,
    rf_preds,
    rf_probs
)
print("✓ Random Forest completed")


Training Random Forest...

Random Forest training completed.

✓ Random Forest completed


## Cell 6 — XGBoost

In [6]:
xgb_model = train_xgboost(
    X_train_smote,
    y_train_smote
)
xgb_preds, xgb_probs = generate_predictions(
    xgb_model,
    X_test
)
comparison_engine.add_model_result(
    "XGBoost",
    y_test,
    xgb_preds,
    xgb_probs
)
print("✓ XGBoost completed")


Training XGBoost...

XGBoost training completed.

✓ XGBoost completed


## Cell 7 — LightGBM

In [7]:
lgbm_model = train_lightgbm(
    X_train_smote,
    y_train_smote
)
lgbm_preds, lgbm_probs = generate_predictions(
    lgbm_model,
    X_test
)
comparison_engine.add_model_result(
    "LightGBM",
    y_test,
    lgbm_preds,
    lgbm_probs
)
print("✓ LightGBM completed")


Training LightGBM...

[LightGBM] [Info] Number of positive: 54138, number of negative: 54138
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005801 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3206
[LightGBM] [Info] Number of data points in the train set: 108276, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
LightGBM training completed.

✓ LightGBM completed


## Cell 8 — CatBoost

In [8]:
cat_model = train_catboost(
    X_train_smote,
    y_train_smote
)
cat_preds, cat_probs = generate_predictions(
    cat_model,
    X_test
)
comparison_engine.add_model_result(
    "CatBoost",
    y_test,
    cat_preds,
    cat_probs
)
print("✓ CatBoost completed")


Training CatBoost...

CatBoost training completed.

✓ CatBoost completed


## Cell 9 — Comparison Table

In [9]:
comparison_df = comparison_engine.get_comparison_table()
comparison_df

,Model,F1_Macro,F1_Weighted,Precision,Recall,ROC_AUC
0,LightGBM,0.525917,0.874411,0.303754,0.061549,0.698357
1,XGBoost,0.516875,0.873136,0.287449,0.049101,0.694451
2,CatBoost,0.484056,0.870085,0.333333,0.007607,0.693420
3,Random Forest,0.517676,0.874173,0.328638,0.048409,0.676221
4,Logistic Regression,0.442316,0.674158,0.097782,0.448133,0.548626


## Cell 10 — Save Comparison Table

In [10]:
comparison_engine.save_results()
print("✓ Model comparison results saved to reports/tables/")


Model comparison saved to:
reports/tables/model_comparison.csv

✓ Model comparison results saved to reports/tables/


## Cell 11 — SHAP Analysis

In [11]:
X_shap_sample = X_test.sample(
    n=1000,
    random_state=42
)
explainer = create_tree_explainer(
    xgb_model
)
shap_explanation, shap_values = compute_shap_values(
    explainer,
    X_shap_sample
)
print("✓ SHAP analysis completed")


Creating SHAP explainer...


Computing SHAP values...

SHAP computation completed.



ValueError: too many values to unpack (expected 2)

## Cell 12 — SHAP Summary Plot

In [ ]:
plot_shap_summary(
    shap_values,
    X_shap_sample
)
save_current_figure(
    "shap_summary_plot.png"
)
print("✓ SHAP summary plot saved to reports/figures/")

## Cell 13 — Temporal SHAP

In [ ]:
feature_columns = X_train.columns.tolist()
yearly_shap_df = compute_yearly_shap_importance(
    model=xgb_model,
    df=df,
    feature_columns=feature_columns,
    encoders=encoders,
    sample_size=300
)
print("✓ Temporal SHAP analysis completed")

## Cell 14 — Temporal Drift Plot

In [ ]:
plot_temporal_feature_drift(
    yearly_shap_df,
    top_n=5
)
save_current_figure(
    "temporal_shap_drift.png"
)
print("✓ Temporal drift plot saved to reports/figures/")

## Cell 15 — Save Temporal SHAP Table

In [ ]:
yearly_shap_df.to_csv(
    "reports/tables/yearly_shap_importance.csv",
    index=False
)
print("Temporal SHAP table saved successfully to reports/tables/")

## Optional Cell 16 — Generate Final Report Summary

In [ ]:
# Print final summary
print("\n" + "="*60)
print("FINAL PIPELINE EXECUTION SUMMARY")
print("="*60)

# Best model
best_model = comparison_df.iloc[0]['Model']
best_f1 = comparison_df.iloc[0]['F1 Score (Weighted)']
print(f"\n🏆 Best Model: {best_model}")
print(f"   Weighted F1 Score: {best_f1:.4f}")

# Top 3 features from SHAP
print(f"\n📊 Top 3 Most Important Features (from SHAP):")
top_features = compute_mean_shap_importance(shap_values, X_shap_sample)
for i, row in top_features.head(3).iterrows():
    print(f"   {i+1}. {row['Feature']}: {row['Mean |SHAP Value|']:.4f}")

# Temporal drift insights
print(f"\n⏰ Temporal Drift Insights:")
from src.evaluation.drift_analysis import compute_feature_drift
drift_df = compute_feature_drift(yearly_shap_df)
increasing = drift_df[drift_df['drift_category'] == 'increasing'].head(3)
for _, row in increasing.iterrows():
    print(f"   ↑ {row['feature']}: trend = {row['trend_coefficient']:.4f}")

print("\n" + "="*60)
print("✓ All results saved to reports/ directory")
print("="*60)